# PlantSR ×4 Super-Resolution

**Peng Wang · AAI 521 · Group 1**  
Fine-tune HAT-L and Real-ESRGAN, compare them on the frozen PlantSR splits, and export the super-resolution stage for the team’s Flask app.

**In Colab**, choose a GPU under **Runtime → Change runtime type**, then select **Runtime → Run all**. The default `auto` mode uses the completed checkpoints on Drive. If either is missing, it fine-tunes that model and evaluates the result. The final cells verify the exported ZIP and open an upload form.

| Stage | Input and output |
|---|---|
| Training | One seeded 256 × 256 HR crop per full-size RGB training image per epoch. Bicubic downsampling produces its 64 × 64 LR input. |
| Validation and test | The saved 256 × 256 center crop at the recorded coordinates. All methods receive the same 64 × 64 input. |
| Application | One uploaded image or an intermediate image from a teammate’s stage. The output covers the entire image at 4× width and height. |

## 1. Install dependencies

Run this cell first in a Colab GPU runtime. Pillow 11.3.0 reproduces the original bicubic degradation.

In [1]:
import subprocess
import sys
from importlib.metadata import version

# Keep Colab's matching torch/torchvision pair and its installed pandas and Hub client.
requirements = [
    "Pillow==11.3.0", "ImageHash==4.3.2", "scikit-image==0.25.2",
    "spandrel==0.4.2", "lpips==0.1.4", "safetensors", "huggingface_hub",
    "Flask>=3,<4", "pandas", "matplotlib", "tqdm",
]
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *requirements], check=True)
loaded_pillow = sys.modules.get("PIL")
if loaded_pillow is not None and loaded_pillow.__version__ != version("Pillow"):
    raise RuntimeError("Pillow was already loaded. Choose Runtime → Restart session, then Run all once more.")
print("Pillow", version("Pillow"), "| spandrel", version("spandrel"), "| torch", version("torch"))

Pillow 11.3.0 | spandrel 0.4.2 | torch 2.14.1+cu130


## 2. Connect Drive and choose a run

Check `PROJECT_DIR` and choose the workflow. The original Drive folder names are used to find your existing data and weights.

| Workflow | What runs |
|---|---|
| `auto` | Package the saved models. If weights are missing, use the prepared export (or rebuild it from `PlantSR_dataset/train` and `test`), train the missing models, then evaluate. |
| `evaluate` | Recompute validation and test scores using saved checkpoints and the frozen dataset. |
| `train` | Start from the pinned Hugging Face weights or resume this notebook’s own training run. Evaluate and export both models. |

Fresh training needs a GPU. The first pretrained comparison and LPIPS evaluation download their weights. The verified handoff is saved as `sr_x4_handoff/PlantSR_x4_app.zip` beside the `prepared/` folder.

In [2]:
from pathlib import Path
from tempfile import mkdtemp
import os

from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/AAI521/Final_project")
PREPARED_DIR = PROJECT_DIR / "PlantSR_prepared_opaque_v2_center256" / "prepared"
RUN_DIR = PREPARED_DIR.parent / "sr_x4_pillow_v1" / "runs" / "finetune_3models_l1_v1"
WORK_DIR = PREPARED_DIR.parent / "sr_x4_handoff"
TRAIN_DIR = WORK_DIR / "training"

WORKFLOW = "auto"  #@param ["auto", "evaluate", "train"]
START_PREVIEW = True  #@param {type:"boolean"}
SHOW_PREVIEWS = True  #@param {type:"boolean"}

# Configure the deterministic GPU workspace before the first CUDA operation.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.use_deterministic_algorithms(True, warn_only=True)
torch.set_num_threads(min(torch.get_num_threads(), 4))

if WORKFLOW not in {"auto", "evaluate", "train"}:
    raise ValueError(f"Unknown workflow: {WORKFLOW}")
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f"Mount the project Drive folder or edit PROJECT_DIR: {PROJECT_DIR}")

original_files = {"hat_l": "best_epoch_002.pt", "real_esrgan": "best_epoch_010.pt"}
saved_models = {
    name: (RUN_DIR / name / filename).is_file() or (TRAIN_DIR / name / "selection.json").is_file()
    for name, filename in original_files.items()
}
SR_MODE = {"auto": "auto", "evaluate": "saved", "train": "train"}[WORKFLOW]
RUN_DATA = WORKFLOW != "auto" or not all(saved_models.values())
RUN_VALIDATION = RUN_TEST = RUN_DATA
WORK_DIR.mkdir(parents=True, exist_ok=True)
BUILD_DIR = Path(mkdtemp(prefix="plantsr_app_"))
PACKAGE_DIR = BUILD_DIR / "plantsr_sr"
print(f"Workflow: {WORKFLOW} | device: {DEVICE} | evaluation: {'on' if RUN_VALIDATION else 'skipped'}")
print("Previously saved model selections:", saved_models)
print("New outputs:", WORK_DIR)

Mounted at /content/drive
Workflow: auto | device: cuda | evaluation: skipped
Previously saved model selections: {'hat_l': True, 'real_esrgan': True}
New outputs: /content/drive/MyDrive/AAI521/Final_project/PlantSR_prepared_opaque_v2_center256/sr_x4_handoff


## PlantSR · EDA and Center-Crop Preprocessing

Reuse the team’s frozen export when available. With the official source folders, the same audit and seed rebuild full-size training images and fixed 256 × 256 validation and test crops.

### Define image loading and shared helpers

Decode and orient each original, allow fully opaque alpha, and identify exact duplicates by RGB pixel hash.

In [3]:
import csv
import hashlib
import html
import json
import math
import platform
import random
from collections import Counter, defaultdict
from importlib.metadata import version as prep_package_version
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw, ImageOps
from IPython.display import display
from types import SimpleNamespace
from tempfile import TemporaryDirectory
import shutil

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp"}
AUDIT_SCHEMA_VERSION = 1
PREPROCESSING_POLICY = "allow_fully_opaque_alpha_v2"
REVIEW_FIELDS = ["id_a", "id_b", "phash_distance", "official_split_a",
                 "official_split_b", "decision"]


def audit_error(message):
    raise ValueError(message)


def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def prep_stable_key(*parts):
    return hashlib.sha256(json.dumps(parts, ensure_ascii=False).encode()).hexdigest()


def prep_write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")


def prep_write_csv(path, rows, fields):
    with Path(path).open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def prep_read_csv(path):
    with Path(path).open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))


def preprocessing_environment():
    return {"python": platform.python_version(),
            **{name: prep_package_version(name) for name in ("Pillow", "numpy", "ImageHash", "scipy")}}


def load_source_rgb(path):
    # load() really decodes the image; merely opening a header is not sufficient.
    with Image.open(path) as source:
        if getattr(source, "n_frames", 1) != 1:
            audit_error("unsupported_multiframe")
        if source.mode in {"I", "F"} or source.mode.startswith("I;16"):
            audit_error("unsupported_high_bit_depth")
        if "A" in source.getbands() or "transparency" in source.info:
            # Accept redundant alpha channels; reject actual transparent pixels.
            alpha = source.convert("RGBA").getchannel("A")
            if alpha.getextrema()[0] < 255:
                audit_error("unsupported_transparency")
        source.load()
        orientation = source.getexif().get(274, 1)
        metadata = {"original_mode": source.mode, "original_width": source.width,
                    "original_height": source.height, "exif_orientation": orientation}
        corrected = ImageOps.exif_transpose(source).convert("RGB")
        corrected.load()
        return corrected, metadata


def prep_pixel_hash(image):
    h = hashlib.sha256(f"RGB:{image.width}:{image.height}:".encode())
    h.update(image.tobytes())
    return h.hexdigest()


def prep_find_files(root):
    # Ignore macOS resource forks / hidden metadata directories.
    files = []
    ignored = []
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        rel = path.relative_to(root)
        if any(part.startswith(".") or part == "__MACOSX" for part in rel.parts):
            continue
        (files if path.suffix.lower() in IMAGE_EXTENSIONS else ignored).append(path)
    return files, ignored


def prep_quantiles(values):
    if not values:
        return None
    return dict(zip(("min", "q25", "median", "q75", "max"),
                    map(float, np.percentile(values, [0, 25, 50, 75, 100]))))


def prep_contact_sheet(out, records, seed):
    chosen = sorted(records, key=lambda r: prep_stable_key(seed, r["sample_id"]))[:36]
    if not chosen:
        return
    sheet = Image.new("RGB", (6 * 180, math.ceil(len(chosen) / 6) * 160), "white")
    draw = ImageDraw.Draw(sheet)
    for i, record in enumerate(chosen):
        with Image.open(out / "thumbnails" / (record["sample_id"] + ".jpg")) as thumb:
            thumb.thumbnail((176, 132))
            x, y = (i % 6) * 180, (i // 6) * 160
            sheet.paste(thumb, (x, y))
            draw.text((x + 2, y + 134), record["sample_id"], fill="black")
    sheet.save(out / "sample_contact_sheet.jpg", quality=90)


def prep_review_html(out, pairs, records):
    by_id = {r["sample_id"]: r for r in records}
    body = ["<!doctype html><meta charset='utf-8'><title>PlantSR duplicate review</title>",
            "<style>body{font:16px sans-serif;margin:24px}article{display:flex;gap:20px;"
            "border-top:1px solid #ccc;padding:12px}figure{margin:0;width:320px}"
            "img{max-width:300px;height:220px;object-fit:contain}code{overflow-wrap:anywhere}</style>",
            "<h1>PlantSR near-duplicate review</h1><p>Fill decision in near_duplicate_review.csv: "
            "<b>same_source</b> or <b>distinct</b>. Similar leaves alone are not duplicates. "
            "Open originals for uncertain cases. Hash distance is not a probability.</p>"]
    for pair in pairs:
        body.append(f"<article><p>Distance: {pair['phash_distance']}</p>")
        for key in ("id_a", "id_b"):
            row = by_id[pair[key]]
            sid = row["sample_id"]
            body.append(f"<figure><img src='thumbnails/{sid}.jpg'><figcaption><b>{sid}</b><br>"
                        f"{html.escape(row['official_split'])}<br><code>"
                        f"{html.escape(row['relative_path'])}</code></figcaption></figure>")
        body.append("</article>")
    if not pairs:
        body.append("<p>No candidates at the selected threshold. This does not prove all images are independent.</p>")
    (out / "near_duplicates.html").write_text("\n".join(body), encoding="utf-8")

### Define the audit and duplicate screening

Screen official train and test files for unsupported formats, small images, exact matches, and perceptual-hash candidates.

In [4]:
def audit_source_images(args):
    import imagehash
    roots = {"official_train": args.train_dir.resolve(), "official_test": args.test_dir.resolve()}
    out = args.out.resolve()
    a, b = roots.values()
    if not all(p.is_dir() for p in roots.values()):
        audit_error("Both TRAIN_DIR and TEST_DIR must exist.")
    if a == b or a in b.parents or b in a.parents:
        audit_error("Official train/test directories must be separate, non-nested folders.")
    if any(out == p or p in out.parents or out in p.parents for p in roots.values()):
        audit_error("OUTPUT_DIR must be outside, and must not contain, both source directories.")
    if out.exists() and any(out.iterdir()):
        audit_error("Audit output is not empty. Use a new OUTPUT_DIR directory to preserve previous decisions.")
    if args.crop_size < 1 or not 0 < args.val_fraction < 1 or not 0 <= args.phash_threshold <= 64:
        audit_error("Use positive crop size, 0 < val fraction < 1, and 0 <= pHash threshold <= 64.")
    discovered = {split: prep_find_files(root) for split, root in roots.items()}
    if any(not item[0] for item in discovered.values()):
        audit_error("No images found in an official directory; check the extracted HR folder paths.")
    out.mkdir(parents=True, exist_ok=True)
    (out / "thumbnails").mkdir()
    records, ignored = [], []
    for split, root in roots.items():
        files, other_files = discovered[split]
        ignored.extend({"official_split": split, "path": str(p.relative_to(root))} for p in other_files)
        for index, path in enumerate(files):
            rel = path.relative_to(root).as_posix()
            sid = ("tr_" if split == "official_train" else "te_") + prep_stable_key(split, rel)[:16]
            row = {"sample_id": sid, "official_split": split, "relative_path": rel,
                   "source_path": str(path), "file_bytes": path.stat().st_size,
                   "file_sha256": file_sha256(path), "status": "ok", "reason": ""}
            try:
                rgb, metadata = load_source_rgb(path)
                row.update(metadata, width=rgb.width, height=rgb.height)
                if min(rgb.size) < args.crop_size:
                    row.update(status="excluded", reason="smaller_than_crop")
                else:
                    row.update(pixel_sha256=prep_pixel_hash(rgb), phash=str(imagehash.phash(rgb)))
                    tiny = rgb.copy()
                    tiny.thumbnail((128, 128))
                    means = np.asarray(tiny, dtype=np.float32).mean(axis=(0, 1))
                    row.update(zip(("mean_R_approx", "mean_G_approx", "mean_B_approx"), map(float, means)))
                    rgb.thumbnail((300, 220))
                    rgb.save(out / "thumbnails" / f"{sid}.jpg", quality=85)
            except (OSError, ValueError, SyntaxError, Image.DecompressionBombError) as exc:
                row.update(status="excluded", reason=f"decode_or_format_error: {exc}")
            records.append(row)
            if (index + 1) % 100 == 0:
                print(f"Audited {split}: {index + 1}/{len(files)}", flush=True)
    if len({r["sample_id"] for r in records}) != len(records):
        audit_error("Unexpected sample ID collision.")
    valid = [r for r in records if r["status"] == "ok"]
    exact = defaultdict(list)
    for row in valid:
        exact[row["pixel_sha256"]].append(row)
    exact_rows = []
    for digest, group in sorted(exact.items()):
        for row in group if len(group) > 1 else []:
            exact_rows.append({"pixel_sha256": digest, "sample_id": row["sample_id"],
                               "official_split": row["official_split"], "relative_path": row["relative_path"]})
    pairs = []
    hashes = {r["sample_id"]: int(r["phash"], 16) for r in valid}
    for i, left in enumerate(valid):
        for right in valid[i + 1:]:
            if left["pixel_sha256"] == right["pixel_sha256"]:
                continue
            distance = (hashes[left["sample_id"]] ^ hashes[right["sample_id"]]).bit_count()
            if distance <= args.phash_threshold:
                pairs.append({"id_a": left["sample_id"], "id_b": right["sample_id"],
                              "phash_distance": distance, "official_split_a": left["official_split"],
                              "official_split_b": right["official_split"], "decision": ""})
    pairs.sort(key=lambda p: (p["phash_distance"], p["id_a"], p["id_b"]))
    config = {"schema_version": AUDIT_SCHEMA_VERSION, "preprocessing_policy": PREPROCESSING_POLICY,
              "seed": args.seed, "crop_size": args.crop_size,
              "val_fraction": args.val_fraction, "phash_threshold": args.phash_threshold,
              "roots": {k: str(v) for k, v in roots.items()}, "environment": preprocessing_environment()}
    payload = {"config": config, "records": records, "candidate_pairs": pairs}
    prep_write_json(out / "audit.json", payload)
    fields = sorted(set().union(*(r.keys() for r in records)))
    prep_write_csv(out / "inventory.csv", records, fields)
    prep_write_csv(out / "exact_duplicates.csv", exact_rows,
              ["pixel_sha256", "sample_id", "official_split", "relative_path"])
    prep_write_csv(out / "near_duplicate_review.csv", pairs, REVIEW_FIELDS)
    summaries = {}
    for split in roots:
        all_rows = [r for r in records if r["official_split"] == split]
        ok = [r for r in all_rows if r["status"] == "ok"]
        summaries[split] = {"discovered": len(all_rows), "eligible_before_deduplication": len(ok),
                           "excluded_reasons": dict(Counter(r["reason"] for r in all_rows if r["status"] != "ok")),
                           "original_modes": dict(Counter(r.get("original_mode", "unknown") for r in all_rows)),
                           "width_px": prep_quantiles([r["width"] for r in ok]),
                           "height_px": prep_quantiles([r["height"] for r in ok]),
                           "aspect_ratio": prep_quantiles([r["width"] / r["height"] for r in ok]),
                           "file_bytes": prep_quantiles([r["file_bytes"] for r in all_rows]),
                           "image_mean_RGB_approx": {c: prep_quantiles([r[f"mean_{c}_approx"] for r in ok]) for c in "RGB"}}
    counts = {k: v["discovered"] for k, v in summaries.items()}
    warnings = []
    if counts != {"official_train": 830, "official_test": 200}:
        warnings.append("Source counts differ from expected 830/200. Verify official HR folders.")
    prep_write_json(out / "EDA_summary.json", {"by_official_split": summaries, "warnings": warnings,
               "exact_duplicate_groups": sum(len(g) > 1 for g in exact.values()),
               "near_duplicate_candidates": len(pairs), "ignored_non_image_files": ignored,
               "notes": ["RGB summaries use thumbnails; these are not model normalization statistics.",
                         "No model selection should use test-image content or test metrics."]})
    prep_review_html(out, pairs, valid)
    prep_contact_sheet(out, valid, args.seed)
    print(f"Audit complete: {counts}; {len(pairs)} near-duplicate pairs to review.")
    for warning in warnings:
        print("WARNING:", warning)
    print("Continue with the EDA and duplicate review cells.")

### Define split assignment and center-crop export

Keep official test membership and split related training images as a group. Center each evaluation crop with integer floor division after EXIF correction.

In [5]:
class DuplicateGroups:
    def __init__(self, ids):
        self.parent = {sid: sid for sid in ids}

    def find(self, sid):
        root = sid
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[sid] != sid:
            previous = sid
            sid = self.parent[sid]
            self.parent[previous] = root
        return root

    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b:
            self.parent[max(a, b)] = min(a, b)


def prep_reviewed_groups(payload, review_path):
    valid = {r["sample_id"]: r for r in payload["records"] if r["status"] == "ok"}
    groups = DuplicateGroups(valid)
    first = {}
    for sid, row in valid.items():
        digest = row["pixel_sha256"]
        if digest in first:
            groups.union(sid, first[digest])
        first[digest] = sid
    decisions = {}
    for line, row in enumerate(prep_read_csv(review_path), 2):
        a, b = (row.get("id_a") or "").strip(), (row.get("id_b") or "").strip()
        decision = (row.get("decision") or "").strip().lower()
        if a not in valid or b not in valid or a == b:
            audit_error(f"Invalid sample IDs in review CSV line {line}.")
        if decision not in {"same_source", "distinct"}:
            audit_error(f"Unresolved review at line {line}: choose same_source or distinct.")
        key = tuple(sorted((a, b)))
        if key in decisions and decisions[key] != decision:
            audit_error(f"Conflicting decisions for {key}.")
        decisions[key] = decision
        if decision == "same_source":
            groups.union(a, b)
    required = {tuple(sorted((p["id_a"], p["id_b"]))) for p in payload["candidate_pairs"]}
    if required - decisions.keys():
        audit_error("Candidate rows are missing from review CSV. Restore and review all candidate pairs.")
    for (a, b), decision in decisions.items():
        if decision == "distinct" and groups.find(a) == groups.find(b):
            audit_error(f"Contradictory review: {a} and {b} are linked by same_source/exact matches but marked distinct.")
    return valid, groups


def prep_choose_validation(group_rows, fraction, seed):
    """Choose whole groups nearest to target count, with seeded deterministic order."""
    n = sum(len(v) for v in group_rows.values())
    if len(group_rows) < 2:
        audit_error("Need at least two independent training-source groups for train/validation.")
    target = max(1, min(n - 1, math.floor(n * fraction + 0.5)))
    ordered = sorted(group_rows, key=lambda g: prep_stable_key(seed, "split", g))
    # Subset-sum avoids arbitrary size drift when several originals share one group.
    reachable = {0: None}
    for gid in ordered:
        size = len(group_rows[gid])
        for count in list(reachable):
            if count + size not in reachable:
                reachable[count + size] = (count, gid)
    count = min((c for c in reachable if 0 < c < n), key=lambda c: (abs(c - target), c))
    chosen = set()
    while count:
        count, gid = reachable[count]
        chosen.add(gid)
    return chosen, target


def prep_center_crop_box(width, height, size=256):
    """Return an in-bounds crop centered on the orientation-corrected image."""
    if min(width, height) < size:
        raise ValueError(f"Image {width} x {height} is smaller than crop size {size}.")
    x = (width - size) // 2
    y = (height - size) // 2
    return x, y, x + size, y + size


def build_prepared_export(audit_path, review_path, target_dir):
    """Build in a temporary directory and publish only the complete export."""
    payload = json.loads(audit_path.read_text(encoding="utf-8"))
    config = payload["config"]
    if config["schema_version"] != AUDIT_SCHEMA_VERSION:
        audit_error("Unsupported audit schema; use the script version that created the audit.")
    if config.get("preprocessing_policy") != PREPROCESSING_POLICY:
        audit_error("This audit uses an older image-loading policy. Rerun the audit in a new OUTPUT_DIR.")
    if target_dir.exists():
        audit_error(f"A prepared directory already exists at {target_dir}. Check its summary and manifests.")
    valid, groups = prep_reviewed_groups(payload, review_path)
    # Check both additions/removals and byte changes before trusting the frozen audit.
    for split, root in config["roots"].items():
        files, _ = prep_find_files(Path(root))
        expected = {r["source_path"] for r in payload["records"] if r["official_split"] == split}
        if {str(p) for p in files} != expected:
            audit_error("Source image list changed after audit. Rerun audit into a new directory.")
    for row in payload["records"]:
        if file_sha256(row["source_path"]) != row["file_sha256"]:
            audit_error(f"Source changed after audit: {row['source_path']}. Rerun audit.")
    excluded = [{"sample_id": r["sample_id"], "reason": r["reason"], "reference_id": ""}
                for r in payload["records"] if r["status"] != "ok"]
    components = defaultdict(list)
    for sid, row in valid.items():
        components[groups.find(sid)].append(row)
    train_groups, test_rows, test_related = {}, [], []
    for component in components.values():
        ids = sorted(r["sample_id"] for r in component)
        gid = "g_" + prep_stable_key(*ids)[:16]
        for row in component:
            row["group_id"] = gid
        tests = [r for r in component if r["official_split"] == "official_test"]
        candidates = tests if tests else component
        # Keep test membership; drop any connected official-train samples.
        if tests:
            for row in component:
                if row["official_split"] == "official_train":
                    excluded.append({"sample_id": row["sample_id"], "reason": "same_source_as_official_test",
                                     "reference_id": sorted(r["sample_id"] for r in tests)[0]})
        kept, seen = [], {}
        for row in sorted(candidates, key=lambda r: r["sample_id"]):
            digest = row["pixel_sha256"]
            if digest in seen:
                excluded.append({"sample_id": row["sample_id"], "reason": "exact_pixel_duplicate",
                                 "reference_id": seen[digest]})
            else:
                seen[digest] = row["sample_id"]
                kept.append(row)
        if tests:
            test_rows.extend(kept)
            if len(kept) > 1:
                test_related.append({"group_id": gid, "sample_ids": [r["sample_id"] for r in kept]})
        else:
            train_groups[gid] = kept
    if not test_rows:
        audit_error("No eligible test images remain.")
    val_groups, val_target = prep_choose_validation(train_groups, config["val_fraction"], config["seed"])
    assignments = {"train": [], "val": [], "test": test_rows}
    for gid, rows in train_groups.items():
        assignments["val" if gid in val_groups else "train"].extend(rows)
    seen_groups = {}
    for split, rows in assignments.items():
        for row in rows:
            gid = row["group_id"]
            if gid in seen_groups and seen_groups[gid] != split:
                audit_error("Internal error: a source group crossed splits.")
            seen_groups[gid] = split
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(prefix=".plantsr-export-", dir=target_dir.parent) as tmp:
        staging = Path(tmp) / "prepared"
        staging.mkdir()
        (staging / "manifests").mkdir()
        manifest_fields = ["sample_id", "group_id", "split", "official_split", "relative_path",
                           "width", "height", "file_sha256", "pixel_sha256", "hr_path", "eval_crop_path",
                           "crop_x", "crop_y", "crop_width", "crop_height"]
        for split, rows in assignments.items():
            (staging / "images" / split).mkdir(parents=True)
            if split != "train":
                (staging / "crops" / split).mkdir(parents=True)
            manifest = []
            for row in sorted(rows, key=lambda r: r["sample_id"]):
                rgb, _ = load_source_rgb(row["source_path"])
                if prep_pixel_hash(rgb) != row["pixel_sha256"]:
                    audit_error("Decoded pixels changed since audit. Preserve the environment and rerun audit.")
                sid = row["sample_id"]
                hr_path = f"images/{split}/{sid}.png"
                # Strip EXIF/other metadata: pixels are already corrected and are the target.
                rgb.info.clear()
                rgb.save(staging / hr_path)
                item = {**row, "split": split, "hr_path": hr_path, "eval_crop_path": "",
                        "crop_x": "", "crop_y": "", "crop_width": "", "crop_height": ""}
                if split != "train":
                    size = config["crop_size"]
                    box = prep_center_crop_box(rgb.width, rgb.height, size)
                    crop_path = f"crops/{split}/{sid}.png"
                    rgb.crop(box).save(staging / crop_path)
                    item.update(eval_crop_path=crop_path, crop_x=box[0], crop_y=box[1],
                                crop_width=size, crop_height=size)
                manifest.append(item)
            prep_write_csv(staging / "manifests" / f"{split}.csv", manifest, manifest_fields)
        prep_write_csv(staging / "manifests" / "excluded.csv", excluded, ["sample_id", "reason", "reference_id"])
        counts = {split: len(rows) for split, rows in assignments.items()}
        summary = {"config": config, "build_environment": preprocessing_environment(), "final_counts": counts,
                   "eval_crop_policy": "center",
                   "target_validation_count_after_cleaning": val_target,
                   "excluded_total": len(excluded), "exclusion_counts": dict(Counter(r["reason"] for r in excluded)),
                   "official_test_excluded": [r["sample_id"] for r in excluded if r["sample_id"].startswith("te_")],
                   "related_groups_within_test": test_related, "audit_sha256": file_sha256(audit_path),
                   "review_sha256": file_sha256(review_path),
                   "cross_split_detected_or_reviewed_group_overlap": False,
                   "notes": ["Near-duplicate screening cannot prove no undetected relationships exist.",
                             "Every crop/degradation must inherit its original sample_id and split.",
                             "Test exclusion changes the official benchmark; disclose counts and exclusions.",
                             "Evaluate fixed crops for crop-based results, not full-image benchmark claims."]}
        if sum(counts.values()) + len(excluded) != len(payload["records"]):
            audit_error("Internal error: counts do not reconcile.")
        prep_write_json(staging / "summary.json", summary)
        # A final prepared/ directory only appears after a successful complete build.
        staging.rename(target_dir)
    print("Build complete:", json.dumps(counts), f"; excluded={len(excluded)}")
    print(f"Share these manifests with all teammates: {target_dir / 'manifests'}")
    print("Train: full RGB images, random crops during training. Val/test: fixed HR crops.")
    if summary["official_test_excluded"] or test_related:
        print("WARNING: See summary.json for official-test exclusions / related test groups.")

### Run the audit and inspect the dataset summary

The frozen export runs on its own. If it is missing, audit the official train and test folders or resume the saved audit.

In [6]:
prep_raw = PROJECT_DIR / "PlantSR_dataset"
prep_audit_dir = PROJECT_DIR / "PlantSR_prepared_opaque_v2"
prep_audit_path = prep_audit_dir / "audit.json"
prep_review_path = prep_audit_dir / "near_duplicate_review.csv"
prep_summary_path = PREPARED_DIR / "summary.json"
prep_needs_build = RUN_DATA and not prep_summary_path.is_file()

# Optional inspection and sharing. Leave the galleries off during model selection.
SHOW_PREP_AUDIT = False
SHOW_ALL_VAL_CROPS = False
SHOW_ALL_TEST_CROPS = False
PACKAGE_PREPARED = False

if prep_needs_build:
    prep_roots = {f"official_{s}": str((prep_raw / s).resolve()) for s in ("train", "test")}
    if not all(Path(root).is_dir() for root in prep_roots.values()):
        raise FileNotFoundError(
            f"Provide the frozen export at {PREPARED_DIR} or the official train/ and test/ folders under {prep_raw}."
        )
    if not prep_audit_path.is_file():
        with TemporaryDirectory(prefix=".plantsr-audit-", dir=PROJECT_DIR) as tmp:
            fresh_audit = Path(tmp) / "audit"
            audit_source_images(SimpleNamespace(train_dir=prep_raw / "train", test_dir=prep_raw / "test",
                                  out=fresh_audit, seed=42, crop_size=256,
                                  val_fraction=0.20, phash_threshold=6))
            if prep_audit_dir.exists() and any(prep_audit_dir.iterdir()):
                raise FileExistsError(f"Unfinished audit at {prep_audit_dir}. Move it aside and rerun this cell.")
            fresh_audit.rename(prep_audit_dir)
    prep_payload = json.loads(prep_audit_path.read_text())
    prep_settings = dict(schema_version=AUDIT_SCHEMA_VERSION, preprocessing_policy=PREPROCESSING_POLICY,
                         seed=42, crop_size=256, val_fraction=0.20, phash_threshold=6, roots=prep_roots)
    if not prep_review_path.is_file() or any(
        prep_payload["config"].get(key) != value for key, value in prep_settings.items()
    ):
        raise ValueError(f"Check the audit configuration and its near_duplicate_review.csv in {prep_audit_dir}.")
    print("Audited", len(prep_payload["records"]), "originals.",
          len(prep_payload["candidate_pairs"]), "near-duplicate candidates.")
elif RUN_DATA:
    print(f"Using frozen images and manifests from {PREPARED_DIR}")

### Preview possible near duplicates

Preview candidate pairs when building a fresh dataset. The uploaded PlantSR audit found zero candidates.

In [7]:
if prep_needs_build:
    prep_records = {row["sample_id"]: row for row in prep_payload["records"]}
    for number, row in enumerate(prep_payload["candidate_pairs"], 1):
        print(number, "pHash distance", row["phash_distance"])
        panel = Image.new("RGB", (620, 220), "white")
        for index, field in enumerate(("id_a", "id_b")):
            sid = row[field]
            print(sid, prep_records[sid]["relative_path"])
            with Image.open(prep_audit_dir / "thumbnails" / f"{sid}.jpg") as thumb:
                panel.paste(thumb, (310 * index, 0))
        display(panel)

### Record duplicate-review decisions

For a new audit with candidates, enter each numbered decision as `same_source` or `distinct`. Existing reviewed decisions are retained.

In [8]:
PREP_REVIEW_DECISIONS = {}  # Example: {1: "same_source", 2: "distinct"}
if prep_needs_build:
    candidates = prep_payload["candidate_pairs"]
    rows = prep_read_csv(prep_review_path)
    by_pair = {(row["id_a"], row["id_b"]): row for row in rows}
    for number, decision in PREP_REVIEW_DECISIONS.items():
        if type(number) is not int or not 1 <= number <= len(candidates):
            raise ValueError(f"Invalid pair number {number}")
        if decision not in {"same_source", "distinct"}:
            raise ValueError(f"Invalid decision for pair {number}")
        pair = candidates[number - 1]
        by_pair[pair["id_a"], pair["id_b"]]["decision"] = decision
    if PREP_REVIEW_DECISIONS:
        temporary_review = prep_review_path.with_suffix(".tmp")
        prep_write_csv(temporary_review, rows, REVIEW_FIELDS)
        temporary_review.replace(prep_review_path)
    pending = [index for index, row in enumerate(candidates, 1)
               if by_pair[row["id_a"], row["id_b"]]["decision"] not in {"same_source", "distinct"}]
    if pending:
        raise ValueError(f"Complete the duplicate decisions for pairs {pending} and rerun this cell.")

### Export the prepared dataset and fixed center crops

Export lossless RGB PNGs and the original split manifests. Reusing a frozen export verifies its recorded settings.

In [9]:
if prep_needs_build and not prep_summary_path.is_file():
    build_prepared_export(prep_audit_path, prep_review_path, PREPARED_DIR)
    for sidecar in (prep_audit_path, prep_review_path):
        copy = PREPARED_DIR.parent / sidecar.name
        if copy.resolve() != sidecar.resolve():
            shutil.copy2(sidecar, copy)
if RUN_DATA:
    prep_summary = json.loads(prep_summary_path.read_text())
    needed = dict(seed=42, crop_size=256, val_fraction=0.20, phash_threshold=6,
                  preprocessing_policy=PREPROCESSING_POLICY)
    if prep_summary.get("eval_crop_policy") != "center" or any(
        prep_summary["config"].get(key) != value for key, value in needed.items()
    ):
        raise ValueError(f"The prepared export does not match the shared crop and split policy: {PREPARED_DIR}")
    prep_manifests = {split: prep_read_csv(PREPARED_DIR / "manifests" / f"{split}.csv")
                      for split in ("train", "val", "test")}
    for split, rows in prep_manifests.items():
        if len(rows) != prep_summary["final_counts"][split] or not rows:
            raise ValueError(f"Manifest count differs from summary.json in {split}.")
        for row in rows:
            if split == "train":
                if row["eval_crop_path"]:
                    raise ValueError(f"Unexpected evaluation crop on training image {row['sample_id']}")
                continue
            x, y, _, _ = prep_center_crop_box(int(row["width"]), int(row["height"]), 256)
            recorded = tuple(int(row[key]) for key in ("crop_x", "crop_y", "crop_width", "crop_height"))
            if recorded != (x, y, 256, 256) or not row["eval_crop_path"]:
                raise ValueError(f"Center-crop metadata mismatch for {row['sample_id']}")

### Inspect any excluded originals

Read exclusions from the frozen manifest. Enable `SHOW_PREP_AUDIT` to inspect individual records.

In [10]:
if RUN_DATA and SHOW_PREP_AUDIT:
    prep_excluded = prep_read_csv(PREPARED_DIR / "manifests/excluded.csv")
    for row in prep_excluded:
        print(row["sample_id"], row["reason"], "reference", row["reference_id"])

### Compare the known duplicate pair

The original audit kept `img552.jpg` and excluded its exact RGB duplicate, `img553.jpg`. The optional check uses the raw files when available.

In [11]:
if RUN_DATA and SHOW_PREP_AUDIT:
    pair = [prep_raw / "train" / name for name in ("img552.jpg", "img553.jpg")]
    if all(p.is_file() for p in pair):
        print("Identical RGB pixels:", prep_pixel_hash(load_source_rgb(pair[0])[0]) == prep_pixel_hash(load_source_rgb(pair[1])[0]))

### Verify img271.jpg and its inclusion

The PNG stored as `img271.jpg` has fully opaque alpha. Locate it in the RGB export using the manifests.

In [12]:
if RUN_DATA and SHOW_PREP_AUDIT:
    for split, rows in prep_manifests.items():
        for row in rows:
            if row["official_split"] == "official_train" and row["relative_path"] == "img271.jpg":
                print(f"img271.jpg → {split} / {row['sample_id']}")
                with Image.open(PREPARED_DIR / row["hr_path"]) as img:
                    print("Prepared image", img.size, img.mode)

### Check final counts and locate the shared dataset

Read the actual split counts from `summary.json`. The uploaded run recorded 663 train, 166 validation and 200 test images.

In [13]:
if RUN_DATA:
    print("Split counts", prep_summary["final_counts"])
    print("Exclusions", prep_summary["exclusion_counts"])
    print("Shared manifests", PREPARED_DIR / "manifests")

### Preview 16 fixed evaluation crops

Enable `SHOW_PREP_AUDIT` to display a seeded sample of validation crops. Each title includes its original crop coordinates.

In [14]:
def prep_gallery(rows, title, columns=4):
    import matplotlib.pyplot as plt
    figure, axes = plt.subplots(math.ceil(len(rows)/columns), columns,
                               figsize=(2.7*columns, 2.9*math.ceil(len(rows)/columns)), squeeze=False)
    for axis in axes.flat:
        axis.axis("off")
    for axis, row in zip(axes.flat, rows):
        with Image.open(PREPARED_DIR / row["eval_crop_path"]) as crop:
            axis.imshow(crop.convert("RGB"))
        axis.set_title(f"{row['sample_id']}\n({row['crop_x']}, {row['crop_y']})", fontsize=8)
    figure.suptitle(title)
    figure.tight_layout()
    plt.show()
    plt.close(figure)

if RUN_DATA and SHOW_PREP_AUDIT:
    rows = prep_manifests["val"]
    prep_gallery(random.Random(42).sample(rows, min(16, len(rows))), "Fixed validation crops")

### Display all test center crops

Enable `SHOW_ALL_TEST_CROPS` after final evaluation to inspect the held-out crops.

In [15]:
def prep_all_crops(split):
    rows = prep_manifests[split]
    for start in range(0, len(rows), 25):
        prep_gallery(rows[start:start + 25], f"{split.title()} crops · page {start // 25 + 1}", columns=5)

if RUN_DATA and SHOW_ALL_TEST_CROPS:
    prep_all_crops("test")

### Display all validation center crops

Enable `SHOW_ALL_VAL_CROPS` for a paginated gallery of saved validation crops.

In [16]:
if RUN_DATA and SHOW_ALL_VAL_CROPS:
    prep_all_crops("val")

### Package the prepared dataset for sharing

Enable `PACKAGE_PREPARED` to write the team ZIP with its original `prepared/` layout.

In [17]:
if RUN_DATA and PACKAGE_PREPARED:
    from zipfile import ZIP_STORED, ZipFile
    archive_path = PREPARED_DIR.parent / "PlantSR_prepared_for_team.zip"
    if archive_path.exists():
        print("Existing archive", archive_path)
    else:
        with TemporaryDirectory(dir=archive_path.parent) as tmp:
            staged = Path(tmp) / archive_path.name
            with ZipFile(staged, "w", compression=ZIP_STORED) as archive:
                for path in sorted(PREPARED_DIR.rglob("*")):
                    if path.is_file():
                        archive.write(path, arcname=Path("prepared") / path.relative_to(PREPARED_DIR))
            staged.rename(archive_path)
        print(archive_path)

## 3. Prepare the PlantSR pairs

Read the frozen split manifests. Each validation or test input is a 64 × 64 bicubic reduction of its saved 256 × 256 RGB center crop. The held-out pairs are built in the final evaluation.

### Define reproducible image processing

Use RGB floats for the model and rounded 8-bit RGB predictions for scoring.

In [18]:
from functools import lru_cache
import hashlib
import json
from pathlib import Path
from tempfile import NamedTemporaryFile

import numpy as np
import pandas as pd
import PIL
from PIL import Image
import torch
from skimage.metrics import peak_signal_noise_ratio


def sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def prepared_path(relative):
    path = (PREPARED_DIR / relative).resolve()
    if not path.is_relative_to(PREPARED_DIR.resolve()) or not path.is_file():
        raise FileNotFoundError(path)
    return path


def tensor(image):
    # PIL RGB -> one NCHW float32 image in [0, 1].
    pixels = np.array(image, dtype=np.float32).transpose(2, 0, 1).copy()
    return torch.from_numpy(pixels).div_(255).unsqueeze(0)


def quantize(output):
    # This fixed size applies to the benchmark. The app accepts other dimensions.
    if tuple(output.shape) != (1, 3, 256, 256) or not torch.isfinite(output).all():
        raise ValueError("Expected one finite 256 × 256 RGB benchmark prediction.")
    rgb = output.detach()[0].clamp(0, 1).mul(255).round().to(torch.uint8)
    return Image.fromarray(rgb.permute(1, 2, 0).cpu().numpy())


def psnr(hr, pred):
    target = np.asarray(hr, dtype=np.float32) / 255
    output = np.asarray(pred, dtype=np.float32) / 255
    return float(peak_signal_noise_ratio(target, output, data_range=1.0))

### Check the frozen splits and create fixed validation pairs

Verify split membership and crop coordinates, then create the validation LR inputs.

In [19]:
def read_splits():
    if PIL.__version__ != "11.3.0":
        raise RuntimeError("Run the installation cell, restart the Colab session, and run again for Pillow 11.3.0.")
    summary = json.loads(prepared_path("summary.json").read_text())
    if summary.get("eval_crop_policy") != "center":
        raise ValueError("Select the prepared export with 256 × 256 center crops.")
    frames, ids, groups = {}, set(), set()
    for split in ("train", "val", "test"):
        path = prepared_path(f"manifests/{split}.csv")
        frame = pd.read_csv(path, dtype=str, keep_default_na=False)
        required = {"sample_id", "group_id", "split", "hr_path", "eval_crop_path"}
        if not required.issubset(frame.columns) or frame.empty or set(frame.split) != {split}:
            raise ValueError(f"Invalid {split} manifest: {path}")
        if len(frame) != summary["final_counts"][split]:
            raise ValueError(f"Count mismatch between {path} and summary.json")
        new_ids, new_groups = set(frame.sample_id), set(frame.group_id)
        if (len(new_ids) != len(frame) or "" in new_groups or ids & new_ids or groups & new_groups
                or any(not sid or Path(sid).name != sid or sid in {".", ".."} for sid in new_ids)):
            raise ValueError(f"Duplicate, overlapping or invalid sample/group IDs in {path}.")
        ids.update(new_ids)
        groups.update(new_groups)
        frames[split] = frame
    if len(frames["val"]) != 166 or len(frames["test"]) != 200:
        raise ValueError("This PlantSR benchmark uses 166 validation crops and 200 test crops.")
    return frames


def save_fixed_lr(image, path):
    if path.is_file():
        with Image.open(path) as cached:
            if cached.mode == "RGB" and cached.size == image.size and cached.tobytes() == image.tobytes():
                return
    with NamedTemporaryFile(dir=path.parent, suffix=".png", delete=False) as stream:
        temp = Path(stream.name)
    try:
        image.save(temp, format="PNG")
        temp.replace(path)
    finally:
        temp.unlink(missing_ok=True)


@lru_cache(maxsize=2)
def get_pairs(split):
    if split not in ("val", "test") or not RUN_DATA or (split == "test" and not RUN_TEST):
        raise ValueError("Enable the corresponding evaluation mode to build fixed pairs.")
    folder = WORK_DIR / "pairs" / split
    folder.mkdir(parents=True, exist_ok=True)
    rows = []
    for row in splits[split].itertuples(index=False):
        hr_path = prepared_path(row.eval_crop_path)
        x, y = (int(row.width) - 256) // 2, (int(row.height) - 256) // 2
        if (int(row.crop_x), int(row.crop_y), int(row.crop_width), int(row.crop_height)) != (x, y, 256, 256):
            raise ValueError(f"Unexpected center-crop coordinates for {row.sample_id}.")
        with Image.open(hr_path) as hr:
            if hr.mode != "RGB" or hr.size != (256, 256):
                raise ValueError(f"Expected a saved 256 × 256 RGB crop at {hr_path}.")
            lr = hr.resize((64, 64), Image.Resampling.BICUBIC, reducing_gap=None)
        lr_path = folder / f"{row.sample_id}.png"
        save_fixed_lr(lr, lr_path)
        rows.append(dict(sample_id=row.sample_id, hr_path=str(hr_path), lr_path=str(lr_path),
                         hr_sha256=sha256(hr_path), lr_sha256=sha256(lr_path)))
    return rows


splits = read_splits() if RUN_DATA else {}
train_rows = splits["train"].to_dict("records") if RUN_DATA else []
val_pairs = get_pairs("val") if RUN_DATA else []
if RUN_DATA:
    print("Frozen split counts:", {name: len(rows) for name, rows in splits.items()})

## 4. Load the model architectures

Load HAT-L ×4 and the Real-ESRGAN x4plus generator. Fresh training uses checksum-verified Hugging Face weights. Saved checkpoints reload offline after dependencies are installed.

In [20]:
from pathlib import Path
import sys

PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
(PACKAGE_DIR / '__init__.py').touch()
(PACKAGE_DIR / 'models.py').write_text(r'''"""PlantSR ×4 model loading and inference. Checkpoint reconstruction works offline."""
from __future__ import annotations

from collections.abc import Mapping
import hashlib
from pathlib import Path

import torch
from torch import nn
from torch.utils.checkpoint import checkpoint
from safetensors.torch import load_file
from spandrel import ImageModelDescriptor, ModelLoader

SPECS = {
    "hat_l": {
        "repo_id": "jaideepsingh/upscale_models",
        "revision": "867e7c0ad519aba5d36b5bb4ef4a4a91781914c4",
        "filename": "HAT/HAT-L_SRx4_ImageNet-pretrain.pth",
        "sha256": "5992bd38522f2b8faf11ea4bd8ee08de92465bb66892166576999afc36d60043",
        "architecture": "HAT",
        "scale": 4
    },
    "real_esrgan": {
        "repo_id": "kornia/real_esrgan",
        "revision": "e500e0f0d4f4b0ccb900d1b65442568ef382d6cf",
        "filename": "RealESRGAN_x4plus.safetensors",
        "sha256": "e82eee5fa7456f49025e70d906cd12aef0a90e5360d31d722488e9775156bce4",
        "architecture": "ESRGAN",
        "scale": 4
    }
}
_CHECKPOINT_TARGETS = {"hat_l": ("RHAG", 12), "real_esrgan": ("RRDB", 23)}


def _spec(name: str) -> dict:
    if name not in SPECS:
        raise ValueError(f"Unknown model {name!r}; choose one of {list(SPECS)}.")
    return SPECS[name]


def _verify_sha(path: str | Path, expected: str) -> None:
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    actual = digest.hexdigest()
    if actual != expected:
        raise ValueError(f"Checkpoint SHA256 mismatch for {path}: {actual} != {expected}")


def _unwrap_state(state: Mapping) -> dict:
    """Prefer EMA weights from standard upstream generator checkpoint bundles."""
    while isinstance(state, Mapping):
        nested = next(
            (state[key] for key in ("params_ema", "params", "state_dict", "model_state")
             if key in state and isinstance(state[key], Mapping)),
            None,
        )
        if nested is None:
            break
        state = nested
    if not isinstance(state, Mapping) or not state:
        raise ValueError("The checkpoint does not contain a nonempty state dictionary.")
    if not all(isinstance(key, str) and isinstance(value, torch.Tensor)
               for key, value in state.items()):
        raise ValueError("Model state must map string parameter names to tensors.")
    return dict(state)


def _load_spandrel(name: str, state: Mapping) -> nn.Module:
    descriptor = ModelLoader(device="cpu").load_from_state_dict(_unwrap_state(state))
    if not isinstance(descriptor, ImageModelDescriptor):
        raise ValueError(f"{name}: expected a single-image super-resolution model.")
    expected = _spec(name)
    if descriptor.architecture.id != expected["architecture"] or descriptor.scale != 4:
        raise ValueError(
            f"{name}: got {descriptor.architecture.id} x{descriptor.scale}, "
            f"expected {expected['architecture']} x4."
        )
    if descriptor.input_channels != 3 or descriptor.output_channels != 3:
        raise ValueError(f"{name}: expected three RGB input and output channels.")
    # Bypass descriptor.__call__: it deliberately disables autograd for inference.
    return descriptor.model


def _validate_model(model: nn.Module, name: str) -> None:
    expected = {"hat_l": "HAT", "real_esrgan": "RRDBNet"}[name]
    if type(model).__name__ != expected:
        raise ValueError(f"Expected {expected}, got {type(model).__name__}.")
    if name == "hat_l" and (model.upscale != 4 or len(model.layers) != 12):
        raise ValueError("Expected HAT-L ×4 with 12 RHAG stages.")
    stage, required = _CHECKPOINT_TARGETS[name]
    if sum(type(block).__name__ == stage for block in model.modules()) != required:
        raise ValueError(f"Expected {required} {stage} blocks for {name}.")


def load_original(name: str) -> nn.Module:
    """Fetch a pinned original checkpoint and verify its bytes before loading."""
    from huggingface_hub import hf_hub_download

    spec = _spec(name)
    path = hf_hub_download(repo_id=spec["repo_id"], filename=spec["filename"], revision=spec["revision"])
    _verify_sha(path, spec["sha256"])
    if spec["filename"].endswith(".safetensors"):
        state = load_file(str(path), device="cpu")
    else:
        state = torch.load(path, map_location="cpu", weights_only=True)
    model = _load_spandrel(name, state).float()
    _validate_model(model, name)
    return model


def make_bundle(model: nn.Module, name: str, epoch: int) -> dict:
    """Serialize the raw model with the same keys as the original PlantSR run."""
    _validate_model(model, name)
    return {
        "format_version": 1,
        "model_metadata": {"name": name, "spec": dict(_spec(name))},
        "model_state": model.state_dict(),
        "phase": "fine_tuned",
        "epoch": int(epoch),
    }


def load_bundle(path: str | Path) -> tuple[nn.Module, dict]:
    """Restore an existing PlantSR model on CPU without fetching weights."""
    bundle = torch.load(path, map_location="cpu", weights_only=True)
    metadata = bundle["model_metadata"]
    name = metadata["name"]
    if metadata["spec"] != _spec(name):
        raise ValueError(f"Unexpected model specification in {path}")
    model = _load_spandrel(name, bundle["model_state"]).float()
    _validate_model(model, name)
    return model, bundle


def forward_sr(model: nn.Module, name: str, x: torch.Tensor) -> torch.Tensor:
    """Run the differentiable model; keep predictions unclamped for L1 training."""
    _spec(name)
    if x.ndim != 4 or x.shape[1] != 3:
        raise ValueError(f"Expected an NCHW RGB tensor, received {tuple(x.shape)}.")
    sr = model(x)
    expected = (x.shape[0], 3, x.shape[2] * 4, x.shape[3] * 4)
    if tuple(sr.shape) != expected:
        raise ValueError(f"{name}: output shape {tuple(sr.shape)} differs from {expected}.")
    return sr


def enable_activation_checkpointing(model: nn.Module, name: str) -> int:
    """Checkpoint whole stages while preserving native state-dict parameter names."""
    _validate_model(model, name)
    class_name, expected_count = _CHECKPOINT_TARGETS[name]
    targets = [module for module in model.modules()
               if module.__class__.__name__ == class_name]
    if len(targets) != expected_count:
        raise ValueError(f"{name}: activation-checkpoint target count is incorrect.")

    def install(module):
        if getattr(module, "_plantsr_activation_checkpointing", False):
            return
        original_forward = module.forward

        def checkpointed_forward(*args, **kwargs):
            if module.training and torch.is_grad_enabled():
                return checkpoint(original_forward, *args, use_reentrant=False, **kwargs)
            return original_forward(*args, **kwargs)

        module.forward = checkpointed_forward
        module._plantsr_activation_checkpointing = True

    for module in targets:
        install(module)
    return len(targets)
''', encoding='utf-8')

# Re-executing setup can create a new staging folder in the same runtime.
if str(BUILD_DIR) in sys.path:
    sys.path.remove(str(BUILD_DIR))
sys.path.insert(0, str(BUILD_DIR))
for module_name in list(sys.modules):
    if module_name == 'plantsr_sr' or module_name.startswith('plantsr_sr.'):
        del sys.modules[module_name]
from plantsr_sr import models

## 5. Generate reproducible training crops

Each epoch draws one 256 × 256 HR crop from every full-size training image. The original seed, sample ID and epoch determine the crop. Pillow bicubic produces its 64 × 64 LR input.

In [21]:
import hashlib
import random

from PIL import Image
from torch.utils.data import DataLoader, Dataset


class EpochCrops(Dataset):
    """One reproducible HR/LR pair per training image per epoch."""

    def __init__(self, rows, epoch):
        self.rows = rows
        self.epoch = epoch

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        sid = row["sample_id"]
        with Image.open(prepared_path(row["hr_path"])) as image:
            if image.mode != "RGB" or min(image.size) < 256:
                raise ValueError(f"Expected full-size RGB training image: {sid}")
            digest = hashlib.sha256(f"RGB:{image.width}:{image.height}:".encode())
            digest.update(image.tobytes())
            if digest.hexdigest() != row["pixel_sha256"]:
                raise ValueError(f"Training pixels differ from the frozen manifest: {sid}")
            key = hashlib.sha256(f"42:{self.epoch}:{sid}".encode()).digest()
            rng = random.Random(int.from_bytes(key[:8], "big"))
            x = rng.randint(0, image.width - 256)
            y = rng.randint(0, image.height - 256)
            hr = image.crop((x, y, x + 256, y + 256))
        lr = hr.resize((64, 64), Image.Resampling.BICUBIC, reducing_gap=None)
        return tensor(lr)[0], tensor(hr)[0], sid, x, y


def crop_loader(epoch):
    return DataLoader(
        EpochCrops(train_rows, epoch), batch_size=1, shuffle=True, num_workers=0,
        generator=torch.Generator().manual_seed(42 + epoch),
    )

## 6. Fine-tune HAT-L and Real-ESRGAN

HAT-L trains for 3 epochs and keeps the highest validation PSNR. Real-ESRGAN trains for **10 total epochs** and keeps epoch 10. Both use L1 loss, AdamW at 1e-5 and four-image gradient accumulation. Fresh runs save crop coordinates, validation history and a resumable checkpoint on Drive.

### Define validation and checkpoint verification

Measure each epoch on the fixed validation pairs. The export section checks reloading in a fresh process.

In [22]:
import gc
import json
import math
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm


@torch.inference_mode()
def validation_psnr(model, name):
    model.eval()
    scores = []
    for pair in val_pairs:
        with Image.open(pair["lr_path"]) as lr, Image.open(pair["hr_path"]) as hr:
            prediction = models.forward_sr(model, name, tensor(lr).to(DEVICE))
            scores.append(float(psnr(hr, quantize(prediction))))
    return sum(scores) / len(scores)

### Define the weight-update loop

Each optimizer update combines up to four crops. Record the crop coordinates for each epoch.

In [23]:
def train_one_epoch(model, name, optimizer, epoch, max_updates=None):
    model.train().requires_grad_(True)
    loader = crop_loader(epoch)
    total = len(loader)
    if total == 0:
        raise ValueError("No training images in the manifest.")
    optimizer.zero_grad(set_to_none=True)
    total_loss = 0.0
    updates = 0
    coordinates = []
    for i, (lr, hr, sid, x, y) in enumerate(tqdm(loader, desc=f"{name}, epoch {epoch}")):
        prediction = models.forward_sr(model, name, lr.to(DEVICE))
        loss = torch.nn.functional.l1_loss(prediction, hr.to(DEVICE))
        if not torch.isfinite(loss):
            raise FloatingPointError(f"Non-finite training loss for {name}")
        # Normalize a short final accumulation group by its actual size.
        group_size = min(4, total - (i // 4) * 4)
        (loss / group_size).backward()
        total_loss += loss.detach().item()
        coordinates.append((sid[0], epoch, x.item(), y.item()))
        if (i + 1) % 4 == 0 or i + 1 == total:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            updates += 1
            if max_updates is not None and updates >= max_updates:
                break
    crops = pd.DataFrame(coordinates, columns=["sample_id", "epoch", "crop_x", "crop_y"])
    return total_loss / len(coordinates), crops


def save_torch(path, value):
    temporary = path.with_suffix(".pt.tmp")
    torch.save(value, temporary)
    temporary.replace(path)


def save_table(path, frame):
    temporary = path.with_suffix(".csv.tmp")
    frame.to_csv(temporary, index=False)
    temporary.replace(path)

### Freeze this experiment’s configuration

Bind each resumable checkpoint to its training recipe, split manifests and validation pairs.

In [24]:
def training_identity(name):
    return {
        "model": name, "recipe": "PlantSR-L1-center256-v1", "seed": 42,
        "training_manifest": sha256(prepared_path("manifests/train.csv")),
        "validation_manifest": sha256(prepared_path("manifests/val.csv")),
        "validation_crops": hashlib.sha256(
            json.dumps(sorted((r["sample_id"], r["hr_sha256"], r["lr_sha256"]) for r in val_pairs)).encode()
        ).hexdigest(),
    }


def fit_clean(name, epochs):
    if not RUN_DATA:
        raise ValueError("Full training requires RUN_DATA=True and the frozen prepared dataset.")
    if DEVICE.type != "cuda":
        raise RuntimeError("Choose a GPU in Colab under Runtime → Change runtime type for fine-tuning.")
    folder = TRAIN_DIR / name
    if folder.resolve() == (RUN_DIR / name).resolve():
        raise ValueError("Use a separate training directory to preserve the original checkpoints.")
    folder.mkdir(parents=True, exist_ok=True)
    last = folder / "last.pt"
    identity = training_identity(name)
    torch.manual_seed(42)
    if last.is_file():
        model, previous = models.load_bundle(last)
        if previous.get("training_identity") != identity or previous.get("phase") != "resume":
            raise ValueError(f"Data or training settings differ from {last}. Choose another TRAIN_DIR.")
        history = previous["history"]
    else:
        model = models.load_original(name)
        previous, history = None, []
    model.to(DEVICE)
    models.enable_activation_checkpointing(model, name)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=1e-5, betas=(0.9, 0.99), weight_decay=0.0, foreach=False,
    )
    if previous is not None:
        optimizer.load_state_dict(previous["optimizer_state"])
        torch.set_rng_state(previous["rng_cpu"])
        torch.cuda.set_rng_state(previous["rng_cuda"], device=DEVICE)
        del previous
    best = max((r["val_psnr_rgb_db"] for r in history), default=-float("inf"))
    for epoch in range(len(history) + 1, epochs + 1):
        loss, crops = train_one_epoch(model, name, optimizer, epoch)
        score = validation_psnr(model, name)
        if math.isnan(score):
            raise FloatingPointError(f"Undefined validation PSNR for {name}.")
        history.append({"epoch": epoch, "train_l1": loss, "val_psnr_rgb_db": score})
        bundle = models.make_bundle(model, name, epoch)
        bundle.update(training_identity=identity, val_psnr_rgb_db=score)
        if (name == "hat_l" and score > best) or (name == "real_esrgan" and epoch == epochs):
            save_torch(folder / f"epoch_{epoch:03d}.pt", bundle)
        best = max(best, score)
        save_table(folder / f"crops_epoch_{epoch:03d}.csv", crops)
        resume = dict(bundle, phase="resume", history=history, optimizer_state=optimizer.state_dict(),
                      rng_cpu=torch.get_rng_state(), rng_cuda=torch.cuda.get_rng_state(DEVICE))
        save_torch(last, resume)
        del resume, bundle
        save_table(folder / "history.csv", pd.DataFrame(history))
        print(f"{name} epoch {epoch}: training L1 {loss:.5f}, validation PSNR {score:.5f} dB")

    selected_epoch = (max(history, key=lambda row: row["val_psnr_rgb_db"])["epoch"]
                      if name == "hat_l" else epochs)
    path = folder / f"epoch_{selected_epoch:03d}.pt"
    # last.pt is the source of truth if Colab interrupted a CSV/selection write.
    save_table(folder / "history.csv", pd.DataFrame(history))
    selection = {"epoch": selected_epoch, "total_epochs": epochs, "sha256": sha256(path),
                 "training_identity": identity}
    marker = folder / "selection.json"
    temp = marker.with_suffix(".json.tmp")
    temp.write_text(json.dumps(selection, indent=2) + "\n", encoding="utf-8")
    temp.replace(marker)
    del model, optimizer
    gc.collect()
    torch.cuda.empty_cache()
    return path, selected_epoch

### Run the 50-step smoke tests

Optional GPU check using pretrained weights. The full run starts from its own checkpoint.

In [25]:
RUN_SMOKE = False  #@param {type:"boolean"}

if RUN_SMOKE:
    if not RUN_DATA or DEVICE.type != "cuda":
        raise RuntimeError("Select the train workflow and a Colab GPU for the optional 50-step checks.")
    for name in ("hat_l", "real_esrgan"):
        torch.manual_seed(42)
        model = models.load_original(name).to(DEVICE)
        models.enable_activation_checkpointing(model, name)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, betas=(0.9, 0.99),
                                      weight_decay=0.0, foreach=False)
        probe = list(model.parameters())[-1]
        initial = probe.detach().cpu().clone()
        loss, sampled = train_one_epoch(model, name, optimizer, epoch=1, max_updates=50)
        if (len(sampled) + 3) // 4 != 50 or torch.equal(initial, probe.detach().cpu()):
            raise RuntimeError(f"The 50-step update check failed for {name}.")
        print(f"{name}: 50 optimizer updates, training L1 {loss:.5f}")
        del initial, probe, optimizer, model
        gc.collect()
        torch.cuda.empty_cache()

## 7. Select the application checkpoints

Reuse the original HAT-L epoch 2 and Real-ESRGAN epoch 10, or resolve the checkpoints from a clean run. HAT-L is the app default.

In [26]:
# The original run selected HAT-L on validation. Real-ESRGAN uses exactly epoch 10.
legacy_file = RUN_DIR / "experiment.json"
legacy = json.loads(legacy_file.read_text()) if legacy_file.is_file() and SR_MODE != "train" else {}
protocol_file = RUN_DIR / "final_test_x4/protocol.json"
old_lock = json.loads(protocol_file.read_text()) if protocol_file.is_file() and SR_MODE != "train" else {}
if legacy and old_lock and legacy["run_id"] != old_lock["run_id"]:
    raise ValueError("The saved experiment and its Test protocol have different run IDs.")


def select_checkpoint(name, budget, historical_epoch):
    historical = RUN_DIR / name / f"best_epoch_{historical_epoch:03d}.pt"
    marker = TRAIN_DIR / name / "selection.json"
    if SR_MODE != "train" and historical.is_file():
        if RUN_DATA and legacy:
            for split, expected in legacy["data"]["manifest_sha256"].items():
                if sha256(prepared_path(f"manifests/{split}.csv")) != expected:
                    raise ValueError(f"The prepared {split} manifest differs from the saved run.")
            expected_pairs = {p["sample_id"]: p for p in legacy["data"]["val_pairs"]}
            if (set(expected_pairs) != {p["sample_id"] for p in val_pairs}
                    or any(p["hr_sha256"] != expected_pairs[p["sample_id"]]["hr_sha256"] for p in val_pairs)):
                raise ValueError("The validation reference images differ from the saved run.")
        if old_lock and sha256(historical) != old_lock["checkpoints"][f"{name}_e{historical_epoch}"]:
            raise ValueError(f"Saved checkpoint differs from the frozen Test protocol: {historical}")
        path, epoch, origin = historical, historical_epoch, "original run"
    elif marker.is_file():
        selection = json.loads(marker.read_text())
        if selection["total_epochs"] != budget:
            raise ValueError(f"Training budget differs from {marker}")
        if RUN_DATA and selection["training_identity"] != training_identity(name):
            raise ValueError(f"Training data differ from {marker}")
        epoch = int(selection["epoch"])
        path = marker.parent / f"epoch_{epoch:03d}.pt"
        if sha256(path) != selection["sha256"]:
            raise ValueError(f"Checkpoint differs from {marker}")
        origin = "clean run"
    elif SR_MODE == "saved" or not RUN_DATA:
        raise FileNotFoundError(
            f"Expected {historical} or a completed clean run at {marker.parent}. "
            "Restore your saved models, or enable RUN_DATA and set SR_MODE to 'auto' to fine-tune."
        )
    else:
        print(f"Fine-tuning {name} for {budget} total epochs. Progress is saved in {TRAIN_DIR / name}.")
        path, epoch = fit_clean(name, budget)
        origin = "clean run"

    info = torch.load(path, map_location="cpu", weights_only=True)
    metadata = info["model_metadata"]
    if (info["phase"] != "fine_tuned" or info["epoch"] != epoch or metadata["name"] != name
            or metadata["spec"] != models.SPECS[name] or (name == "real_esrgan" and epoch != 10)
            or (name == "hat_l" and not 1 <= epoch <= budget)):
        raise ValueError(f"Unexpected model, phase, epoch, or weight specification: {path}")
    expected_run = legacy.get("run_id") or old_lock.get("run_id")
    if origin == "original run" and expected_run and info.get("run_id") != expected_run:
        raise ValueError(f"Checkpoint belongs to a different experiment: {path}")
    return path, epoch, origin

### Continue Real-ESRGAN to a total of 10 epochs

Use the completed epoch-10 checkpoint or train to epoch 10. A resumed run continues from the last completed epoch.

In [27]:
selected_paths, selection_sources = {}, {}
for name, budget, historical_epoch, key in (
        ("hat_l", 3, 2, "hat_l"), ("real_esrgan", 10, 10, "real_esrgan_e10")):
    checkpoint_path, epoch, origin = select_checkpoint(name, budget, historical_epoch)
    selected_paths[key] = checkpoint_path
    selection_sources[key] = origin
    if name == "hat_l":
        hat_epoch = epoch
    print(f"{name} | epoch {epoch} | {origin} | {checkpoint_path.name}")

hat_l | epoch 2 | original run | best_epoch_002.pt
real_esrgan | epoch 10 | original run | best_epoch_010.pt


## 8. Validation metrics and saved results

Measure the full 256 × 256 RGB prediction after rounding to 8-bit pixels. Higher PSNR and SSIM and lower AlexNet LPIPS indicate a closer match to the reference. Each evaluation saves per-image scores, their means, matching PNGs, and checkpoint identities.

In [28]:
import gc
from tempfile import mkdtemp

from matplotlib import pyplot as plt
from matplotlib.patches import Rectangle
from skimage.metrics import structural_similarity
from tqdm.auto import tqdm

SCORE_COLUMNS = ["psnr_rgb_db", "ssim_rgb", "lpips_alex"]


def evaluate_split(split):
    import lpips

    pairs = get_pairs(split)
    destination = WORK_DIR / "evaluation" / split
    destination.mkdir(parents=True, exist_ok=True)
    # Separate directories keep interrupted runs from overwriting completed reports.
    report = Path(mkdtemp(prefix="run_", dir=destination))
    pd.DataFrame(pairs).to_csv(report / "pairs.csv", index=False)

    methods = {"bicubic": (None, None)}
    for name, key in [("hat_l", "hat_l"), ("real_esrgan", "real_esrgan_e10")]:
        if key not in selected_paths:
            continue
        if split == "val":
            methods[f"{name}_pretrained"] = (name, None)
        epoch = hat_epoch if name == "hat_l" else 10
        methods[f"{name}_e{epoch}"] = (name, selected_paths[key])

    protocol = {
        "split": split, "n_images": len(pairs),
        "manifest_sha256": sha256(prepared_path(f"manifests/{split}.csv")),
        "pairs_csv_sha256": sha256(report / "pairs.csv"),
        "degradation": f"Pillow {PIL.__version__} RGB bicubic 256 to 64, reducing_gap=None",
        "scoring": "Full quantized RGB8, no border shave. SSIM Gaussian sigma 1.5, population covariance. LPIPS AlexNet v0.1.",
        "weights_sha256": {
            label: (sha256(path) if path else models.SPECS[name]["sha256"] if name else None)
            for label, (name, path) in methods.items()
        }
    }
    (report / "protocol.json").write_text(json.dumps(protocol, indent=2))
    perceptual = lpips.LPIPS(net="alex", version="0.1", verbose=False).to(DEVICE).eval().requires_grad_(False)
    tables = []
    try:
        for label, (name, checkpoint) in methods.items():
            model = None
            if name is not None:
                model = models.load_bundle(checkpoint)[0] if checkpoint else models.load_original(name)
                model.to(DEVICE).eval().requires_grad_(False)
            output_dir = report / "predictions" / label
            output_dir.mkdir(parents=True)
            rows = []
            try:
                with torch.inference_mode():
                    for item in tqdm(pairs, desc=f"{split} · {label}"):
                        with Image.open(item["hr_path"]) as hr, Image.open(item["lr_path"]) as lr:
                            if model is None:
                                prediction = lr.resize(hr.size, Image.Resampling.BICUBIC, reducing_gap=None)
                            else:
                                prediction = quantize(models.forward_sr(model, name, tensor(lr).to(DEVICE)))
                            prediction.save(output_dir / f"{item['sample_id']}.png")
                            ref = np.asarray(hr, dtype=np.float32) / 255
                            out = np.asarray(prediction, dtype=np.float32) / 255
                            ssim = structural_similarity(ref, out, data_range=1.0, channel_axis=-1,
                                                         gaussian_weights=True, sigma=1.5, use_sample_covariance=False)
                            alex = perceptual(tensor(hr).to(DEVICE) * 2 - 1,
                                              tensor(prediction).to(DEVICE) * 2 - 1).item()
                            values = dict(psnr_rgb_db=psnr(hr, prediction), ssim_rgb=float(ssim), lpips_alex=alex)
                        if (np.isnan(values["psnr_rgb_db"]) or np.isneginf(values["psnr_rgb_db"])
                                or not np.isfinite([ssim, alex]).all()):
                            raise FloatingPointError(f"Invalid metrics for {label}, {item['sample_id']}.")
                        rows.append(dict(sample_id=item["sample_id"], method=label, **values))
            finally:
                del model
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            scores = pd.DataFrame(rows)
            scores.to_csv(report / f"{label}.csv", index=False)
            tables.append(scores)
    finally:
        del perceptual
    scores = pd.concat(tables, ignore_index=True)
    summary = scores.groupby("method", sort=False)[SCORE_COLUMNS].mean()
    summary["n_images"] = scores.groupby("method").sample_id.nunique()
    scores.to_csv(report / "per_image.csv", index=False)
    summary.to_csv(report / "summary.csv")
    display(summary.round(5))
    print("Results:", report)
    return report, summary

## 9. Compare validation checkpoints

Compare bicubic, pretrained weights, and the selected fine-tuned checkpoints on the same 166 images. The curves use the recorded validation PSNR for each training epoch.

In [29]:
val_report = None
if RUN_VALIDATION:
    val_report, val_summary = evaluate_split("val")

### Include the pretrained starting point (epoch 0)

Plot the original weights and each available epoch on the validation PSNR curve.

In [30]:
if val_report is not None:
    fig, ax = plt.subplots(figsize=(6.5, 3.5))
    for name, key in [("hat_l", "hat_l"), ("real_esrgan", "real_esrgan_e10")]:
        if key not in selected_paths:
            continue
        history_path = selected_paths[key].parent / "history.csv"
        if not history_path.is_file():
            continue
        history = pd.read_csv(history_path).sort_values("epoch")
        history = history[history.epoch.between(1, 3 if name == "hat_l" else 10)]
        if not history.empty:
            initial = val_summary.loc[f"{name}_pretrained", "psnr_rgb_db"]
            ax.plot([0, *history.epoch], [initial, *history.val_psnr_rgb_db], marker="o", label=name)
    if ax.lines:
        ax.set(xlabel="Epoch", ylabel="Validation RGB PSNR (dB)")
        ax.legend()
        fig.tight_layout()
        fig.savefig(val_report / "learning_curves.png", dpi=160)
        plt.show()
    plt.close(fig)

## 10. Inspect the same region

The upper row shows the full validation crop. The lower row enlarges the same 64 × 64 region in the reference, bicubic image, and fine-tuned predictions.

In [31]:
def show_detail(pair, report, labels, roi):
    paths = {"HR reference": Path(pair["hr_path"])}
    paths.update({label: report / "predictions" / label / f"{pair['sample_id']}.png" for label in labels})
    left, top, right, bottom = roi
    if not (0 <= left < right <= 256 and 0 <= top < bottom <= 256):
        raise ValueError("Choose a display region within the 256 × 256 reference crop.")
    fig, axes = plt.subplots(2, len(paths), figsize=(2.8 * len(paths), 5.5), squeeze=False,
                             constrained_layout=True)
    for column, (label, path) in enumerate(paths.items()):
        with Image.open(path) as image:
            axes[0, column].imshow(image)
            axes[1, column].imshow(image.crop(roi), interpolation="nearest")
        axes[0, column].add_patch(Rectangle((left - .5, top - .5), right - left, bottom - top,
                                            fill=False, edgecolor="gold"))
        axes[0, column].set_title(label.replace("_", " "), fontsize=10)
        for ax in axes[:, column]:
            ax.axis("off")
    fig.suptitle(f"{pair['sample_id']} · ROI {roi}")
    fig.savefig(report / f"detail_{pair['sample_id']}.png", dpi=160)
    plt.show()
    plt.close(fig)


if SHOW_PREVIEWS and val_report is not None:
    example = min(val_pairs, key=lambda item: item["sample_id"])
    labels = [name for name in val_summary.index if not name.endswith("_pretrained")]
    show_detail(example, val_report, labels, (64, 64, 128, 128))

## 11. Final held-out evaluation

With checkpoints fixed, compare bicubic and both trained models on the 200 test crops. The optional previews use three seeded examples and select the display region from the HR reference alone.

**Recorded results from the supplied notebooks** · 200 held-out center crops. The table below is the previous experiment. An evaluation run prints a new table and writes its per-image scores to Drive.

| Method | RGB PSNR ↑ (dB) | RGB SSIM ↑ | LPIPS ↓ |
|---|---:|---:|---:|
| Bicubic ×4 | 29.13934 | 0.78486 | 0.35233 |
| HAT-L · epoch 2 | **32.38102** | **0.85807** | **0.19633** |
| Real-ESRGAN · epoch 10 | 30.71354 | 0.82241 | 0.23219 |

In [32]:
import random

if RUN_TEST:
    test_report, test_summary = evaluate_split("test")
    if SHOW_PREVIEWS:
        by_id = {p["sample_id"]: p for p in get_pairs("test")}
        regions = {}
        for sample_id in random.Random(42).sample(sorted(by_id), 3):
            pair = by_id[sample_id]
            with Image.open(pair["hr_path"]) as ref:
                gray = np.asarray(ref.convert("L"), dtype=np.float32)
            dy, dx = np.gradient(gray)
            magnitude = np.hypot(dx, dy)
            boxes = [(x, y, x + 64, y + 64) for y in range(0, 193, 32) for x in range(0, 193, 32)]
            roi = max(boxes, key=lambda b: magnitude[b[1]:b[3], b[0]:b[2]].mean())
            regions[sample_id] = roi
            show_detail(pair, test_report, test_summary.index, roi)
        (test_report / "test_rois.json").write_text(json.dumps(regions, indent=2))

## 12. Integrate super-resolution with the team app

Use `restore_sr(image)` for one PIL image, `Predictor` for repeated requests, or `make_handler` for the team registry. Uploads retain their full field of view. Overlapping tiles limit GPU memory on larger images.

In [33]:
(PACKAGE_DIR / "inference.py").write_text(r'''"""Offline PlantSR ×4 image inference and the team's Flask registry adapter."""
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageOps

from .models import forward_sr, load_bundle

METHODS = {"hat_l": "HAT-L", "real_esrgan": "Real-ESRGAN"}
SCALE = 4


def as_rgb(image):
    """Orient a PIL upload and composite any transparency over white."""
    if not isinstance(image, Image.Image):
        raise TypeError("Pass a PIL.Image.Image object.")
    image = ImageOps.exif_transpose(image)
    if "A" in image.getbands() or "transparency" in image.info:
        rgba = image.convert("RGBA")
        bg = Image.new("RGBA", rgba.size, "white")
        bg.alpha_composite(rgba)
        return bg.convert("RGB")
    return image.convert("RGB")


def _load(method, package_dir, device):
    if method not in METHODS:
        raise ValueError(f"Choose a method from {list(METHODS)}.")
    root = Path(package_dir) if package_dir else Path(__file__).resolve().parent
    model, bundle = load_bundle(root / "weights" / f"{method}.pt")
    epoch = int(bundle["epoch"])
    if bundle["model_metadata"]["name"] != method or bundle["phase"] != "fine_tuned":
        raise ValueError(f"Unexpected checkpoint for {method}.")
    if method == "real_esrgan" and epoch != 10:
        raise ValueError("Expected the Real-ESRGAN epoch-10 checkpoint.")
    return model.eval().requires_grad_(False).to(device), epoch


def _predict(tile, model, method, device):
    pixels = np.asarray(tile, dtype=np.float32)
    height, width = pixels.shape[:2]
    if method == "hat_l":
        # HAT shifts 16-pixel windows. Edge padding also supports one-pixel inputs.
        pixels = np.pad(pixels, ((0, -height % 16), (0, -width % 16), (0, 0)), mode="edge")
    batch = torch.from_numpy(pixels.transpose(2, 0, 1).copy()).unsqueeze(0).div_(255).to(device)
    output = forward_sr(model, method, batch)
    if not torch.isfinite(output).all():
        raise FloatingPointError(f"{method} produced non-finite pixels.")
    output = output[0, :, :height * SCALE, :width * SCALE].clamp(0, 1).mul(255).round().byte()
    image = Image.fromarray(output.permute(1, 2, 0).contiguous().cpu().numpy())
    return image, (pixels.shape[1], pixels.shape[0])


def _restore(image, predict):
    """Return a full-field RGB image and the largest padded tile size."""
    image = as_rgb(image)
    width, height = image.size
    core, halo = 128, 16
    if max(width, height) <= core:
        return predict(image)
    output = Image.new("RGB", (SCALE * width, SCALE * height))
    largest = (0, 0)
    for top in range(0, height, core):
        for left in range(0, width, core):
            right, bottom = min(left + core, width), min(top + core, height)
            x, y = max(0, left - halo), max(0, top - halo)
            patch = image.crop((x, y, min(width, right + halo), min(height, bottom + halo)))
            up, size = predict(patch)
            largest = (max(largest[0], size[0]), max(largest[1], size[1]))
            inner = (SCALE * (left-x), SCALE * (top-y), SCALE * (right-x), SCALE * (bottom-y))
            output.paste(up.crop(inner), (SCALE * left, SCALE * top))
    return output, largest


class Predictor:
    """Load one checkpoint once for repeated requests."""
    def __init__(self, method="hat_l", *, package_dir=None, device=None):
        self.method = method
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        self.model, self.epoch = _load(method, package_dir, self.device)

    @torch.inference_mode()
    def with_tile_size(self, image):
        return _restore(image, lambda tile: _predict(tile, self.model, self.method, self.device))

    def __call__(self, image):
        return self.with_tile_size(image)[0]


def restore_sr(image, method="hat_l", *, package_dir=None, device=None):
    """Upscale one PIL image. Use Predictor to reuse the model across images."""
    return Predictor(method, package_dir=package_dir, device=device)(image)


def make_handler(method="hat_l", *, package_dir=None, device=None):
    """Build a lazy TaskHandler for the shared Flask registry."""
    from src.inference.interface import TaskHandler

    if method not in METHODS:
        raise ValueError(f"Choose a method from {list(METHODS)}.")

    class PlantSRHandler(TaskHandler):
        task = "super_resolution"
        uses_gpu = True

        def __init__(self):
            super().__init__()
            self.method = method
            self.label = f"{METHODS[method]} ×4 (PlantSR)"
            self.description = "Upscales the full image. Large inputs use overlapping tiles."
            self.predictor = None
            self.version = method

        def load(self):
            self.predictor = Predictor(method, package_dir=package_dir, device=device)
            self.version = f"{method}-epoch-{self.predictor.epoch}"
            self._loaded = True

        def unload(self):
            self.predictor = None
            self._loaded = False

        def run(self, image, mask=None, seed=42):
            if self.predictor is None:
                self.load()
            return self.predictor.with_tile_size(image)

    return PlantSRHandler()
''', encoding="utf-8")

5229

## 13. Add image upload and model selection

The Flask demo accepts PNG and JPEG uploads. It shows the original, the ×4 result, and matching center details from bicubic and model output.

In [34]:
(BUILD_DIR / "demo.py").write_text(r'''"""Launch the standalone module with python demo.py."""
import base64
import io
from threading import Lock

from flask import Flask, render_template_string, request
from PIL import Image, UnidentifiedImageError

from plantsr_sr.inference import METHODS, Predictor, as_rgb

app = Flask(__name__)
app.config["MAX_CONTENT_LENGTH"] = 8 * 1024 * 1024
_lock = Lock()
_predictor = None
PAGE = """<!doctype html>
<html lang="en"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>PlantSR · 4× super-resolution</title><style>
body{font:1rem/1.5 system-ui,sans-serif;max-width:960px;margin:2rem auto;padding:0 1rem;color:#1c3524}
form,section{display:flex;flex-wrap:wrap;gap:1rem}form{align-items:end}label{display:grid;gap:.3rem}
figure{min-width:0;flex:1 1 240px;margin:1rem 0}img{display:block;max-width:100%;margin:.35rem 0}
</style><main><h1>PlantSR · 4× super-resolution</h1>
<form method="post" enctype="multipart/form-data" onsubmit="document.getElementById('status').hidden=false">
<label>Input image<input type="file" accept="image/png,image/jpeg" name="image" required></label>
<label>Model<select name="method">{% for key,name in methods.items() %}
<option value="{{key}}" {% if key==selected %}selected{% endif %}>{{name}}</option>{% endfor %}</select></label>
<button>Restore image</button></form><p>PNG or JPEG, up to 1024 pixels per side and an 8 MiB upload limit.</p>
<p id="status" role="status" hidden>Processing your image. Large images use multiple tiles.</p>
{% if error %}<p role="alert">{{error}}</p>{% endif %}
{% if result %}<section><figure><figcaption>Input · {{w}} × {{h}}</figcaption><img src="{{original}}" alt="Input"></figure>
<figure><figcaption>{{methods[selected]}} · {{w*4}} × {{h*4}}</figcaption><img id="restored" src="{{result}}" alt="Restored image"></figure></section>
<a id="download" download="plantsr_4x.png" href="{{result}}">Download full-resolution PNG</a>
<h2>Same center detail at output scale</h2><section>
<figure><figcaption>Bicubic ×4</figcaption><img src="{{bicubic_detail}}" alt="Bicubic detail"></figure>
<figure><figcaption>{{methods[selected]}} ×4</figcaption><img src="{{sr_detail}}" alt="Model detail"></figure>
</section>{% endif %}</main></html>"""


def png_url(image):
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode("ascii")


def page(method="hat_l", status=200, **context):
    return render_template_string(PAGE, methods=METHODS, selected=method, **context), status


@app.errorhandler(413)
def too_large(_error):
    return page(status=413, error="The upload exceeds 8 MiB.")


@app.route("/", methods=["GET", "POST"])
def index():
    global _predictor
    if request.method == "GET":
        return page()
    method, upload = request.form.get("method", "hat_l"), request.files.get("image")
    if method not in METHODS or upload is None:
        return page(status=400, error="Choose a listed model and an image.")
    try:
        with Image.open(upload.stream, formats=("JPEG", "PNG")) as raw:
            if max(raw.size) > 1024:
                return page(method, 400, error="Each input dimension must be at most 1024 pixels.")
            original = as_rgb(raw)
    except (UnidentifiedImageError, OSError, ValueError, Image.DecompressionBombError):
        return page(method, 400, error="Choose a readable PNG or JPEG.")
    with _lock:
        if _predictor is None or _predictor.method != method:
            _predictor = None
            _predictor = Predictor(method)
        restored = _predictor(original)
    width, height = restored.size
    dw, dh = min(192, width), min(192, height)
    x, y = (width - dw)//2, (height - dh)//2
    box = (x, y, x + dw, y + dh)
    # Match the full-image bicubic coordinate grid while allocating only the detail window.
    bbox_lr = tuple(v / 4 for v in box)
    detail = original.resize((dw, dh), Image.Resampling.BICUBIC, box=bbox_lr, reducing_gap=None)
    return page(method, w=original.width, h=original.height, original=png_url(original),
                result=png_url(restored), bicubic_detail=png_url(detail), sr_detail=png_url(restored.crop(box)))


if __name__ == "__main__":
    app.run(host="127.0.0.1", port=7860, debug=False)
''', encoding="utf-8")

4289

## 14. Export the application bundle

Export both fine-tuned checkpoints, the offline model loader, the Flask demo, and integration instructions to `PlantSR_x4_app.zip`.

In [35]:
import sys
from importlib.metadata import version
import json
import zipfile
from pathlib import Path
import torch

weights_dir = PACKAGE_DIR / "weights"
weights_dir.mkdir(exist_ok=True)
app_manifest = {"default": "hat_l", "scale": 4, "models": {}}
for method, key, epoch in (("hat_l", "hat_l", hat_epoch), ("real_esrgan", "real_esrgan_e10", 10)):
    checkpoint = selected_paths[key]
    original = torch.load(checkpoint, map_location="cpu", weights_only=True)
    if (original["phase"], original["model_metadata"]["name"], int(original["epoch"])) != ("fine_tuned", method, epoch):
        raise ValueError(f"Unexpected model or epoch in {checkpoint}")
    # Preserve inference weights while excluding training configuration and history.
    portable = {k: original[k] for k in ("model_state", "model_metadata", "phase", "epoch")}
    portable["format_version"] = 1
    target = weights_dir / f"{method}.pt"
    torch.save(portable, target)
    app_manifest["models"][method] = {
        "epoch": epoch, "file": f"weights/{target.name}", "sha256": sha256(target),
        "source_checkpoint": Path(checkpoint).name, "source_sha256": sha256(checkpoint),
    }
    del original, portable
(PACKAGE_DIR / "manifest.json").write_text(json.dumps(app_manifest, indent=2), encoding="utf-8")

# Record the software versions actually used. Wheel indexes differ between CPU and CUDA.
packages = ("torch", "torchvision", "spandrel", "safetensors", "numpy", "Pillow", "Flask")
requirements = [f"{pkg}=={version(pkg).split('+')[0]}" for pkg in packages]
(BUILD_DIR / "requirements.txt").write_text("\n".join(requirements) + "\n", encoding="utf-8")
(BUILD_DIR / "README.md").write_text("""# PlantSR ×4 super-resolution

Extract the ZIP. Create a Python environment matching the notebook environment (see manifest below).
Install the matching PyTorch and torchvision builds for your CPU or CUDA runtime, then run:

    python -m pip install -r requirements.txt
    python demo.py

Open http://127.0.0.1:7860. Choose HAT-L (default) or Real-ESRGAN (epoch 10).
The weights are included. GPU inference is faster for large inputs.

## Integrating with Flask

Copy plantsr_sr/ beside src/ in the team repository. Register the SR handlers
after the bicubic_x4 baseline inside build_registry() in src/inference/factory.py:

    from plantsr_sr.inference import make_handler
    registry.register(make_handler("hat_l"), default=True)
    registry.register(make_handler("real_esrgan"))

Both handlers use task super_resolution. Their run(image, mask=None, seed=42)
method returns a tuple (RGB PIL image, largest model-input tile size). Use the
first element as the image for the next restoration step. The full field of view
is preserved and both dimensions increase fourfold. A spatial mask used after
this stage must be mapped to the output dimensions with nearest-neighbor resizing.

For direct inference without the team registry:

    from PIL import Image
    from plantsr_sr.inference import Predictor
    restore = Predictor("hat_l")
    with Image.open("input.png") as image:
        restored = restore(image)
    restored.save("output.png")

The ZIP contains inference code, two fine-tuned weight files, and
plantsr_sr/manifest.json with epochs and checkpoint hashes.
""" + f"\nTested notebook interpreter: Python {sys.version.split()[0]}.\n", encoding="utf-8")

archive = WORK_DIR / "PlantSR_x4_app.zip"
contents = ["README.md", "requirements.txt", "demo.py", "plantsr_sr/__init__.py",
            "plantsr_sr/models.py", "plantsr_sr/inference.py", "plantsr_sr/manifest.json",
            "plantsr_sr/weights/hat_l.pt", "plantsr_sr/weights/real_esrgan.pt"]
stage = archive.with_suffix(".part")
with zipfile.ZipFile(stage, "w", zipfile.ZIP_STORED, allowZip64=True) as bundle_zip:
    for relative in contents:
        bundle_zip.write(BUILD_DIR / relative, relative)
stage.replace(archive)
print(f"Team handoff file ({archive.stat().st_size / 2**20:.0f} MiB): {archive}")

Team handoff file (222 MiB): /content/drive/MyDrive/AAI521/Final_project/PlantSR_prepared_opaque_v2_center256/sr_x4_handoff/PlantSR_x4_app.zip


## 15. Verify the application

Extract the ZIP into a fresh directory and test a real Flask upload with each saved checkpoint. The check verifies the RGB PNG download and its ×4 dimensions.

In [36]:
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path

verify_code = r'''import io
import base64
from html.parser import HTMLParser
import sys

import numpy as np
import torch
from PIL import Image
from demo import app
from plantsr_sr.inference import METHODS, _restore

torch.set_num_threads(min(torch.get_num_threads(), 4))


class Downloads(HTMLParser):
    def __init__(self):
        super().__init__()
        self.urls = []

    def handle_starttag(self, tag, attrs):
        attrs = dict(attrs)
        if tag == "a" and "download" in attrs:
            self.urls.append(attrs.get("href", ""))


# Check the tiling grid separately with a known pixel-exact 4× reference.
rng = np.random.default_rng(42)
odd = Image.fromarray(rng.integers(0, 256, (173, 211, 3), dtype=np.uint8))
def nearest(img):
    return img.resize((4 * img.width, 4 * img.height), Image.Resampling.NEAREST), img.size
assert _restore(odd, nearest)[0].tobytes() == nearest(odd)[0].tobytes()
if len(sys.argv) > 1:
    with Image.open(sys.argv[1]) as ref:
        probe = ref.convert("RGB")
else:
    probe = Image.fromarray(rng.integers(0, 256, (64, 64, 3), dtype=np.uint8))
assert probe.size == (64, 64)
app.config["TESTING"] = True
with app.test_client() as client:
    assert client.get("/").status_code == 200
    for method in METHODS:
        buf = io.BytesIO()
        probe.save(buf, format="PNG")
        buf.seek(0)
        response = client.post("/", data={"method": method, "image": (buf, "probe.png")})
        assert response.status_code == 200, (method, response.status_code)
        links = Downloads()
        links.feed(response.get_data(as_text=True))
        assert len(links.urls) == 1 and links.urls[0].startswith("data:image/png;base64,")
        payload = base64.b64decode(links.urls[0].split(",", 1)[1], validate=True)
        with Image.open(io.BytesIO(payload)) as image:
            image.load()
            assert (image.format, image.mode, image.size) == ("PNG", "RGB", (256, 256))
        print(f"{method}: 64 × 64 input → 256 × 256 downloadable RGB PNG", flush=True)
'''
verification_input = []
if globals().get("val_pairs"):
    verification_input = [str(Path(val_pairs[0]["lr_path"]).resolve())]
with tempfile.TemporaryDirectory(prefix="plantsr_verify_") as extracted:
    with zipfile.ZipFile(archive) as bundle_zip:
        bundle_zip.extractall(extracted)
    probe_run = subprocess.run([sys.executable, "-u", "-c", verify_code, *verification_input],
                               cwd=extracted, capture_output=True, text=True)
print(probe_run.stdout)
if probe_run.returncode:
    raise RuntimeError(probe_run.stderr or "Flask bundle verification failed.")
print("Verified team handoff:", archive)

hat_l: 64 × 64 input → 256 × 256 downloadable RGB PNG
real_esrgan: 64 × 64 input → 256 × 256 downloadable RGB PNG

Verified team handoff: /content/drive/MyDrive/AAI521/Final_project/PlantSR_prepared_opaque_v2_center256/sr_x4_handoff/PlantSR_x4_app.zip


## 16. Preview the Flask interface in Colab

Set `START_PREVIEW = True` in setup to open the upload form while the Colab session runs.

In [37]:
if START_PREVIEW:
    import subprocess
    import sys
    import time
    from urllib.request import urlopen
    from google.colab import output

    # A rerun should serve the current weights and demo.
    previous = globals().get("demo_process")
    if previous is not None and previous.poll() is None:
        previous.terminate()
        previous.wait(timeout=10)
    demo_process = subprocess.Popen([sys.executable, "demo.py"], cwd=BUILD_DIR)
    for _ in range(90):
        if demo_process.poll() is not None:
            raise RuntimeError("Flask exited during startup. Check the Colab runtime log.")
        try:
            with urlopen("http://127.0.0.1:7860/", timeout=1) as response:
                if response.status == 200:
                    break
        except OSError:
            time.sleep(1)
    else:
        demo_process.terminate()
        raise TimeoutError("The Flask preview did not start.")
    output.serve_kernel_port_as_iframe(7860, height=800)

<IPython.core.display.Javascript object>